# Few-shot flower classification

Scored over many random episodes. Compares four ways to classify 5-way 1-shot and 5-way 5-shot episodes drawn from
40 Oxford Flowers-102 species that no model below is trained on:
  1. an untrained ResNet-18 (random weights) as the embedding, classified by prototypes
  2. an ImageNet-pretrained ResNet-18, frozen, classified by prototypes
  3. the same frozen embedding with a logistic regression fitted on the support set
  4. the pretrained ResNet-18 after episodic prototypical training on the other 62 species

Needs: torch, torchvision, scikit-learn, scipy (torchvision downloads Flowers-102).
Run the cells in order. Use a GPU runtime for the full run.

In [14]:
import argparse
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from torch.utils.data import ConcatDataset, DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.models import ResNet18_Weights, resnet18

## Settings
Options keep their defaults in a notebook; pass a list to `parse_known_args([...])` to change them, e.g. `["--dry-run"]` for a quick check with random images.

In [15]:
parser = argparse.ArgumentParser()
parser.add_argument("--data", default="./data")
parser.add_argument("--episodes", type=int, default=600, help="test episodes per setting")
parser.add_argument("--train-episodes", type=int, default=2000)
parser.add_argument("--workers", type=int, default=0,
                    help="DataLoader worker processes; 0 loads in the notebook process (workers forked in a notebook can hang)")
parser.add_argument("--dry-run", action="store_true", help="random images, no downloads")
args, _ = parser.parse_known_args()  # ignore Jupyter's own -f kernel.json argument

SEED = 0
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
start = time.time()

## Data
Pools Flowers-102's train, val and test splits (or random images with `--dry-run`).

In [16]:
mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                              transforms.ToTensor(), transforms.Normalize(mean, std)])
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
                               transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])

def flowers(transform):
    # Pool the official train, val and test splits: the episodes below need every species
    splits = [datasets.Flowers102(args.data, split=s, transform=transform, download=True)
              for s in ("train", "val", "test")]
    labels = np.concatenate([np.asarray(d._labels) for d in splits])
    return ConcatDataset(splits), labels

if args.dry_run:
    g = torch.Generator().manual_seed(SEED)
    labels = np.repeat(np.arange(102), 25)
    images = torch.randn(len(labels), 3, 64, 64, generator=g)
    eval_set = train_set = TensorDataset(images, torch.as_tensor(labels))
    args.episodes, args.train_episodes = 20, 5
else:
    eval_set, labels = flowers(eval_tf)
    train_set, _ = flowers(train_tf)

Split the 102 species into 62 base species (for training) and 40 novel species (for testing).

In [17]:
rng = np.random.default_rng(SEED)
species = rng.permutation(102)
base_classes, novel_classes = np.sort(species[:62]), np.sort(species[62:])
by_class = {c: np.flatnonzero(labels == c) for c in range(102)}
print(f"device: {device}  images: {len(labels)}  "
      f"base species: {len(base_classes)}  novel species: {len(novel_classes)}")

device: cuda  images: 8189  base species: 62  novel species: 40


## Episodes
An episode picks `n_way` species, `k_shot` support images and `n_query` query images per species.

In [18]:
def sample_episode(rng, classes, n_way, k_shot, n_query):
    """Pick n_way classes; return support and query image indices with labels 0..n_way-1."""
    chosen = rng.choice(classes, n_way, replace=False)
    s_idx, s_lab, q_idx, q_lab = [], [], [], []
    for j, c in enumerate(chosen):
        picks = rng.choice(by_class[c], k_shot + n_query, replace=False)
        s_idx += picks[:k_shot].tolist(); s_lab += [j] * k_shot
        q_idx += picks[k_shot:].tolist(); q_lab += [j] * n_query
    return s_idx, torch.tensor(s_lab), q_idx, torch.tensor(q_lab)

## Model
ResNet-18 embeddings and prototype scoring.

In [19]:
def backbone(pretrained):
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Identity()  # keep the 512-dimensional embedding, drop ImageNet's classifier
    return model.to(device)

@torch.no_grad()
def embed(model, indices):
    model.eval()
    loader = DataLoader(Subset(eval_set, indices), batch_size=256, num_workers=args.workers)
    return torch.cat([model(x.to(device)).cpu() for x, _ in loader])

def prototype_logits(f_support, y_support, f_query, n_way):
    # Snell et al. (2017): each prototype is the mean embedding of its class's support images;
    # a query scores each class by negative squared Euclidean distance to its prototype
    protos = torch.stack([f_support[y_support == j].mean(0) for j in range(n_way)])
    return -torch.cdist(f_query, protos) ** 2

## Evaluation
Every method is scored on the same random episodes from the novel species.

In [20]:
novel_idx = np.concatenate([by_class[c] for c in novel_classes])
row_of = {i: r for r, i in enumerate(novel_idx)}  # image index -> row in the feature table

def evaluate(features, classifier, n_way, k_shot, n_query=15):
    rng = np.random.default_rng(SEED + 1)  # same episodes for every method
    accs = []
    for _ in range(args.episodes):
        s, ys, q, yq = sample_episode(rng, novel_classes, n_way, k_shot, n_query)
        fs, fq = features[[row_of[i] for i in s]], features[[row_of[i] for i in q]]
        if classifier == "prototypes":
            pred = prototype_logits(fs, ys, fq, n_way).argmax(1)
        else:
            fs, fq = F.normalize(fs, dim=1), F.normalize(fq, dim=1)
            clf = LogisticRegression(max_iter=1000).fit(fs.numpy(), ys.numpy())
            pred = torch.as_tensor(clf.predict(fq.numpy()))
        accs.append((pred == yq).float().mean().item())
    accs = np.array(accs) * 100
    return accs.mean(), 1.96 * accs.std() / np.sqrt(len(accs))  # mean and 95% interval

results = {}
def run(name, model, classifier):
    t0 = time.time()
    features = embed(model, novel_idx)
    t1 = time.time()
    results[name] = [evaluate(features, classifier, 5, k) for k in (1, 5)]
    print(f"{name}: embedding {t1 - t0:.0f} s, episodes {time.time() - t1:.0f} s", flush=True)

## Baselines
Untrained and ImageNet-pretrained ResNet-18, with prototypes or logistic regression.

In [21]:
run("Untrained ResNet-18 + prototypes", backbone(False), "prototypes")
pretrained = backbone(True if not args.dry_run else False)
run("ImageNet ResNet-18 + prototypes", pretrained, "prototypes")
run("ImageNet ResNet-18 + logistic regression", pretrained, "logistic")

Untrained ResNet-18 + prototypes: embedding 88 s, episodes 6 s
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 80.3MB/s]


ImageNet ResNet-18 + prototypes: embedding 91 s, episodes 8 s
ImageNet ResNet-18 + logistic regression: embedding 96 s, episodes 93 s


## Episodic training on base species
Set up the episode sampler, model and optimizer.

In [22]:
class EpisodeSampler:
    """Yields one training episode (5-way, 5-shot, 10 queries per class) per batch."""
    def __init__(self, n_episodes, n_way=5, k_shot=5, n_query=10):
        self.n, self.n_way, self.k, self.q = n_episodes, n_way, k_shot, n_query
        self.rng = np.random.default_rng(SEED + 2)
    def __len__(self):
        return self.n
    def __iter__(self):
        for _ in range(self.n):
            s, _, q, _ = sample_episode(self.rng, base_classes, self.n_way, self.k, self.q)
            yield s + q

sampler = EpisodeSampler(args.train_episodes)
loader = DataLoader(train_set, batch_sampler=sampler, num_workers=args.workers)
model = backbone(True if not args.dry_run else False)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
n_way, k_shot, n_query = sampler.n_way, sampler.k, sampler.q
y_support = torch.arange(n_way).repeat_interleave(k_shot).to(device)
y_query = torch.arange(n_way).repeat_interleave(n_query).to(device)

Train with the prototypical loss, one episode per step.

In [23]:
model.train()
for step, (images, _) in enumerate(loader, 1):
    # sample_episode orders images class by class, so these label tensors line up with the batch
    feats = model(images.to(device))
    f_support, f_query = feats[: n_way * k_shot], feats[n_way * k_shot:]
    loss = F.cross_entropy(prototype_logits(f_support, y_support, f_query, n_way), y_query)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 250 == 0 or step == len(sampler):
        print(f"  training episode {step}/{len(sampler)}  loss {loss.item():.3f}")

  training episode 250/2000  loss 0.000
  training episode 500/2000  loss 0.000
  training episode 750/2000  loss 0.018
  training episode 1000/2000  loss 0.237
  training episode 1250/2000  loss 0.000
  training episode 1500/2000  loss 0.287
  training episode 1750/2000  loss 0.000
  training episode 2000/2000  loss 0.000


Evaluate the trained model on the novel species.

In [24]:
run("ImageNet ResNet-18 + episodic training + prototypes", model, "prototypes")

ImageNet ResNet-18 + episodic training + prototypes: embedding 83 s, episodes 7 s


## Report

In [25]:
print(f"\nAccuracy on {len(novel_classes)} unseen species, mean of {args.episodes} episodes "
      f"(95% interval), 15 queries per class")
print(f"{'method':<54}{'5-way 1-shot':>16}{'5-way 5-shot':>16}")
for name, ((m1, c1), (m5, c5)) in results.items():
    print(f"{name:<54}{m1:>9.1f} ± {c1:<4.1f}{m5:>9.1f} ± {c5:<4.1f}")
gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(f"\n{gpu}, torch {torch.__version__}, torchvision "
      f"{__import__('torchvision').__version__}, {time.time() - start:.0f} s")


Accuracy on 40 unseen species, mean of 600 episodes (95% interval), 15 queries per class
method                                                    5-way 1-shot    5-way 5-shot
Untrained ResNet-18 + prototypes                           36.2 ± 0.8      45.6 ± 0.7 
ImageNet ResNet-18 + prototypes                            76.0 ± 0.9      95.5 ± 0.3 
ImageNet ResNet-18 + logistic regression                   83.7 ± 0.8      96.1 ± 0.3 
ImageNet ResNet-18 + episodic training + prototypes        81.0 ± 0.8      97.9 ± 0.2 

Tesla T4, torch 2.14.1+cu130, torchvision 0.29.1+cu130, 4515 s


## Extra measurements
Everything below reuses the definitions above (`SEED`, the species split, `sample_episode`, `backbone`,
`embed`, `prototype_logits`, `evaluate`, `EpisodeSampler`, the transforms, `args.workers = 0`).
It does not touch the earlier cells, their outputs, or the variables `model`, `sampler`, `loader`,
`optimizer` and `results`.

- **A.** Shots and ways grid for the three frozen-embedding methods.
- **B.** Method 4 retrained from scratch, evaluated at step 0 and every 250 episodes.
- **C.** The same grid as A for the step-2,000 model from B.

### Setup
Checks that the kernel still holds the notebook's definitions. If it stops with a list of missing
names, re-run the definition cells above: Imports, Settings, Data, the species-split cell, Episodes,
Model, Evaluation, and the training-setup cell (it defines `EpisodeSampler`; it builds a model but
does not train). Do not re-run Baselines or the training loop.

In [26]:
import csv, json, os

needed = ["SEED", "args", "device", "eval_set", "train_set", "eval_tf", "train_tf", "labels",
          "base_classes", "novel_classes", "by_class", "sample_episode", "backbone", "embed",
          "prototype_logits", "novel_idx", "row_of", "evaluate", "EpisodeSampler"]
missing = [n for n in needed if n not in globals()]
if missing:
    raise RuntimeError(f"The kernel no longer has: {', '.join(missing)}. Re-run the definition cells "
                       "listed above this cell (not Baselines or the training loop), then this cell.")
assert args.workers == 0, "keep args.workers = 0: worker processes hang in this notebook setup"
if args.dry_run:
    print("NOTE: --dry-run is on (random images); these numbers mean nothing.")

FULL_EPISODES, CHECK_EPISODES, N_QUERY = 600, 20, 15
SETTINGS = [(5, 1), (5, 2), (5, 5), (5, 10), (5, 20), (10, 1), (10, 5), (20, 1), (20, 5)]
smallest = min(len(by_class[c]) for c in novel_classes)
assert smallest >= max(k for _, k in SETTINGS) + N_QUERY, f"a novel species has only {smallest} images"

M_UNTRAINED = "Untrained ResNet-18 + prototypes"
M_PROTO = "ImageNet ResNet-18 + prototypes"
M_LOGREG = "ImageNet ResNet-18 + logistic regression"
M_EPISODIC = "ImageNet ResNet-18 + episodic training + prototypes"

extra = {"A_check": [], "A": [], "B": [], "C": []}
extra_start = time.time()

def setting_name(n_way, k_shot):
    return f"{n_way}-way {k_shot}-shot"

def pm(mean, ci):
    return f"{mean:.1f} ± {ci:.1f}"

def run_grid(part, method, features, classifier, n_episodes):
    """evaluate() on every setting in SETTINGS; returns one row per setting."""
    saved, args.episodes = args.episodes, n_episodes  # evaluate() reads args.episodes
    rows = []
    try:
        for n_way, k_shot in SETTINGS:
            t0 = time.time()
            m, ci = evaluate(features, classifier, n_way, k_shot, n_query=N_QUERY)
            rows.append(dict(part=part, method=method, setting=setting_name(n_way, k_shot),
                             n_way=n_way, k_shot=k_shot, episodes=n_episodes,
                             mean=float(m), ci=float(ci), seconds=time.time() - t0))
            print(f"  {method:<52}{setting_name(n_way, k_shot):<15}{pm(m, ci):>12}"
                  f"{time.time() - t0:>8.1f} s", flush=True)
    finally:
        args.episodes = saved
    extra[part] += rows
    return rows

def print_grid(rows, title):
    methods = list(dict.fromkeys(r["method"] for r in rows))
    cols = [setting_name(n, k) for n, k in SETTINGS]
    cell = {(r["method"], r["setting"]): pm(r["mean"], r["ci"]) for r in rows}
    print(title)
    print(f"{'method':<52}" + "".join(f"{c:>15}" for c in cols))
    for m in methods:
        print(f"{m:<52}" + "".join(f"{cell.get((m, c), '-'):>15}" for c in cols))

print(f"device: {device}  novel species: {len(novel_classes)}  smallest novel species: {smallest} images")

device: cuda  novel species: 40  smallest novel species: 40 images


### A. Shots and ways, frozen embeddings
Embeds the novel images once with each network (about 90 s each) and reuses the features for the whole grid.
Re-seeding with `SEED` first gives the same untrained network as method 1 in Baselines.

In [27]:
t0 = time.time()
torch.manual_seed(SEED)  # the state the Settings cell left, so this is method 1's random network
untrained_net = backbone(False)
feats_untrained = embed(untrained_net, novel_idx)
imagenet_net = backbone(True if not args.dry_run else False)
feats_imagenet = embed(imagenet_net, novel_idx)
del untrained_net, imagenet_net
A_METHODS = [(M_UNTRAINED, feats_untrained, "prototypes"),
             (M_PROTO, feats_imagenet, "prototypes"),
             (M_LOGREG, feats_imagenet, "logistic")]
print(f"embedded {len(novel_idx)} novel images twice in {time.time() - t0:.0f} s")

embedded 3293 novel images twice in 189 s


Quick check: the whole grid on 20 episodes per setting (logistic regression takes most of the time).

In [28]:
for method, feats, classifier in A_METHODS:
    run_grid("A_check", method, feats, classifier, CHECK_EPISODES)
print()
print_grid(extra["A_check"], f"Quick check, {CHECK_EPISODES} episodes per setting (not the real result)")

  Untrained ResNet-18 + prototypes                    5-way 1-shot     37.1 ± 3.2     0.6 s
  Untrained ResNet-18 + prototypes                    5-way 2-shot     40.6 ± 3.5     0.7 s
  Untrained ResNet-18 + prototypes                    5-way 5-shot     44.8 ± 4.1     0.3 s
  Untrained ResNet-18 + prototypes                    5-way 10-shot    44.2 ± 3.9     0.7 s
  Untrained ResNet-18 + prototypes                    5-way 20-shot    47.1 ± 3.3     0.1 s
  Untrained ResNet-18 + prototypes                    10-way 1-shot    23.8 ± 2.4     0.2 s
  Untrained ResNet-18 + prototypes                    10-way 5-shot    31.0 ± 2.0     0.3 s
  Untrained ResNet-18 + prototypes                    20-way 1-shot    15.2 ± 1.4     0.1 s
  Untrained ResNet-18 + prototypes                    20-way 5-shot    23.3 ± 1.1     0.1 s
  ImageNet ResNet-18 + prototypes                     5-way 1-shot     76.7 ± 5.7     0.1 s
  ImageNet ResNet-18 + prototypes                     5-way 2-shot     90.9 ± 2.

Full grid: 600 episodes per setting. Expect roughly 10–20 minutes, mostly logistic regression at 20-way.

In [29]:
extra["A"] = []
for method, feats, classifier in A_METHODS:
    run_grid("A", method, feats, classifier, FULL_EPISODES)
print()
print_grid(extra["A"], f"A. Accuracy on {len(novel_classes)} novel species, {FULL_EPISODES} episodes, "
                       f"{N_QUERY} queries per class (95% interval)")

# Sanity check: the 5-way 1-shot and 5-shot cells should reproduce the Baselines table
if "results" in globals():
    for method in (M_UNTRAINED, M_PROTO, M_LOGREG):
        if method in results:
            new = [r["mean"] for r in extra["A"] if r["method"] == method and r["n_way"] == 5 and r["k_shot"] in (1, 5)]
            old = [float(m) for m, _ in results[method]]
            print(f"  vs Baselines, {method}: 1-shot {new[0]:.2f} (was {old[0]:.2f}), 5-shot {new[1]:.2f} (was {old[1]:.2f})")

  Untrained ResNet-18 + prototypes                    5-way 1-shot     36.2 ± 0.8     5.3 s
  Untrained ResNet-18 + prototypes                    5-way 2-shot     40.8 ± 0.8     6.1 s
  Untrained ResNet-18 + prototypes                    5-way 5-shot     45.6 ± 0.7     5.2 s
  Untrained ResNet-18 + prototypes                    5-way 10-shot    48.3 ± 0.7     3.9 s
  Untrained ResNet-18 + prototypes                    5-way 20-shot    49.5 ± 0.7     7.7 s
  Untrained ResNet-18 + prototypes                    10-way 1-shot    24.5 ± 0.4     4.5 s
  Untrained ResNet-18 + prototypes                    10-way 5-shot    31.9 ± 0.4     7.4 s
  Untrained ResNet-18 + prototypes                    20-way 1-shot    16.6 ± 0.3     8.4 s
  Untrained ResNet-18 + prototypes                    20-way 5-shot    22.6 ± 0.2     8.5 s
  ImageNet ResNet-18 + prototypes                     5-way 1-shot     76.0 ± 0.9    12.3 s
  ImageNet ResNet-18 + prototypes                     5-way 2-shot     89.2 ± 0.

### B. Training curve: method 4 retrained from scratch
Same model, sampler, optimizer and loop as the training cells above, under new names (`model_b`,
`sampler_b`, `loader_b`, `optimizer_b`). Before building the model, the cell replays the random-number
draws the earlier run made before its training loop (the networks built in Baselines and their three
`embed()` calls), so the new model and its augmentations start from the same state as the earlier run.
Each checkpoint embeds the novel images (about 90 s) and evaluates 5-way 1-shot and 5-shot on 600 episodes;
it saves and restores the random-number state, so checkpoints do not change the training run.

In [30]:
def dataloader_seed_draw():
    # every DataLoader iteration draws one base seed from the global torch RNG, as embed() did in Baselines
    torch.empty((), dtype=torch.int64).random_()

torch.manual_seed(SEED)
backbone(False); dataloader_seed_draw()                              # Baselines: method 1
backbone(True if not args.dry_run else False)                        # Baselines: methods 2 and 3
dataloader_seed_draw(); dataloader_seed_draw()

sampler_b = EpisodeSampler(args.train_episodes)                      # same SEED + 2 episode stream
loader_b = DataLoader(train_set, batch_sampler=sampler_b, num_workers=args.workers)
model_b = backbone(True if not args.dry_run else False)
optimizer_b = torch.optim.Adam(model_b.parameters(), lr=1e-5)
n_way_b, k_shot_b, n_query_b = sampler_b.n_way, sampler_b.k, sampler_b.q
y_support_b = torch.arange(n_way_b).repeat_interleave(k_shot_b).to(device)
y_query_b = torch.arange(n_way_b).repeat_interleave(n_query_b).to(device)

def checkpoint(step, window_losses):
    """Evaluate model_b on the novel species and summarise the training losses since the last checkpoint."""
    global feats_b
    rng_state = torch.get_rng_state()  # embed() draws from the global RNG; keep training's stream unchanged
    t0 = time.time()
    feats_b = embed(model_b, novel_idx)  # embed() switches to model_b.eval()
    saved, args.episodes = args.episodes, FULL_EPISODES
    try:
        (m1, c1), (m5, c5) = [evaluate(feats_b, "prototypes", 5, k, n_query=N_QUERY) for k in (1, 5)]
    finally:
        args.episodes = saved
    model_b.train()
    torch.set_rng_state(rng_state)
    w = np.array(window_losses, dtype=float)
    row = dict(part="B", method=M_EPISODIC, step=step,
               acc_1shot=float(m1), ci_1shot=float(c1), acc_5shot=float(m5), ci_5shot=float(c5),
               episodes_in_window=len(w),
               loss_mean=float(w.mean()) if len(w) else None,
               loss_median=float(np.median(w)) if len(w) else None,
               loss_max=float(w.max()) if len(w) else None,
               share_loss_below_0001=float((w < 0.001).mean()) if len(w) else None,
               eval_seconds=time.time() - t0, elapsed_seconds=time.time() - train_b_start)
    extra["B"].append(row)
    loss_txt = (f"loss mean {row['loss_mean']:.4f} median {row['loss_median']:.4f} max {row['loss_max']:.3f} "
                f"<0.001: {row['share_loss_below_0001']:.0%}") if len(w) else "no training yet"
    print(f"  step {step:>4}  1-shot {pm(m1, c1)}  5-shot {pm(m5, c5)}  {loss_txt}  "
          f"elapsed {row['elapsed_seconds']:.0f} s", flush=True)

print(f"ready: {len(sampler_b)} training episodes, checkpoints every 250")

ready: 2000 training episodes, checkpoints every 250


Training loop with checkpoints. Expect about 80 minutes: about 67 for training as before, plus 9 checkpoints of about 90 s.

In [31]:
extra["B"] = []
train_b_start = time.time()
checkpoint(0, [])   # step 0: the pretrained model before any training
window = []
model_b.train()
for step, (images, _) in enumerate(loader_b, 1):
    # sample_episode orders images class by class, so these label tensors line up with the batch
    feats = model_b(images.to(device))
    f_support, f_query = feats[: n_way_b * k_shot_b], feats[n_way_b * k_shot_b:]
    loss = F.cross_entropy(prototype_logits(f_support, y_support_b, f_query, n_way_b), y_query_b)
    optimizer_b.zero_grad()
    loss.backward()
    optimizer_b.step()
    window.append(loss.item())
    if step % 250 == 0 or step == len(sampler_b):
        checkpoint(step, window)
        window = []
print(f"training done in {time.time() - train_b_start:.0f} s")

  step    0  1-shot 76.0 ± 0.9  5-shot 95.5 ± 0.3  no training yet  elapsed 102 s
  step  250  1-shot 77.9 ± 0.9  5-shot 97.2 ± 0.2  loss mean 1.2234 median 0.9121 max 6.897 <0.001: 16%  elapsed 740 s
  step  500  1-shot 80.5 ± 0.8  5-shot 97.5 ± 0.2  loss mean 0.8639 median 0.5256 max 6.183 <0.001: 25%  elapsed 1380 s
  step  750  1-shot 81.5 ± 0.8  5-shot 97.7 ± 0.2  loss mean 0.5745 median 0.2414 max 5.264 <0.001: 33%  elapsed 2006 s
  step 1000  1-shot 81.9 ± 0.8  5-shot 97.8 ± 0.2  loss mean 0.4003 median 0.1305 max 2.729 <0.001: 35%  elapsed 2610 s
  step 1250  1-shot 81.7 ± 0.8  5-shot 97.8 ± 0.2  loss mean 0.4148 median 0.1016 max 5.252 <0.001: 36%  elapsed 3260 s
  step 1500  1-shot 82.2 ± 0.8  5-shot 97.9 ± 0.2  loss mean 0.2901 median 0.0079 max 2.405 <0.001: 48%  elapsed 3936 s
  step 1750  1-shot 81.2 ± 0.8  5-shot 97.8 ± 0.2  loss mean 0.2431 median 0.0009 max 2.666 <0.001: 50%  elapsed 4565 s
  step 2000  1-shot 81.0 ± 0.8  5-shot 97.9 ± 0.2  loss mean 0.2193 median 0.00

### C. Shots and ways for the step-2,000 model
Reuses the novel-species features from B's last checkpoint (the final model), so there is no new embedding.

In [32]:
assert extra["B"] and extra["B"][-1]["step"] == len(sampler_b), "run B to the end first"
extra["C"] = []
run_grid("C", M_EPISODIC, feats_b, "prototypes", FULL_EPISODES)

  ImageNet ResNet-18 + episodic training + prototypes 5-way 1-shot     81.0 ± 0.8     3.4 s
  ImageNet ResNet-18 + episodic training + prototypes 5-way 2-shot     93.9 ± 0.4     3.3 s
  ImageNet ResNet-18 + episodic training + prototypes 5-way 5-shot     97.9 ± 0.2     3.7 s
  ImageNet ResNet-18 + episodic training + prototypes 5-way 10-shot    98.7 ± 0.1     6.7 s
  ImageNet ResNet-18 + episodic training + prototypes 5-way 20-shot    99.2 ± 0.1     5.0 s
  ImageNet ResNet-18 + episodic training + prototypes 10-way 1-shot    74.2 ± 0.6     7.0 s
  ImageNet ResNet-18 + episodic training + prototypes 10-way 5-shot    96.3 ± 0.2     4.9 s
  ImageNet ResNet-18 + episodic training + prototypes 20-way 1-shot    65.6 ± 0.5     8.3 s
  ImageNet ResNet-18 + episodic training + prototypes 20-way 5-shot    93.9 ± 0.2    14.8 s


### Report
Prints the tables, compares the step-2,000 result with the earlier run, and saves
`extra_results.json` and `extra_results.csv` in the kernel's working directory.

In [33]:
title = (f"Accuracy on {len(novel_classes)} novel species, {FULL_EPISODES} episodes per setting, "
         f"{N_QUERY} queries per class (mean ± 95% interval)")
print_grid(extra["A"] + extra["C"], "A + C. " + title)

print(f"\nB. Training curve, {M_EPISODIC} (losses are over the episodes since the previous checkpoint)")
print(f"{'step':>5}{'5-way 1-shot':>14}{'5-way 5-shot':>14}{'loss mean':>11}{'median':>9}{'max':>8}"
      f"{'<0.001':>8}{'elapsed s':>11}")
for r in extra["B"]:
    if r["episodes_in_window"]:
        losses = (f"{r['loss_mean']:>11.4f}{r['loss_median']:>9.4f}{r['loss_max']:>8.3f}"
                  f"{r['share_loss_below_0001']:>8.0%}")
    else:
        losses = f"{'-':>11}{'-':>9}{'-':>8}{'-':>8}"
    print(f"{r['step']:>5}{pm(r['acc_1shot'], r['ci_1shot']):>14}{pm(r['acc_5shot'], r['ci_5shot']):>14}"
          f"{losses}{r['elapsed_seconds']:>11.0f}")

final = extra["B"][-1]
EARLIER = {"1-shot": 81.0, "5-shot": 97.9}
d1, d5 = final["acc_1shot"] - EARLIER["1-shot"], final["acc_5shot"] - EARLIER["5-shot"]
print(f"\nStep {final['step']} vs the earlier run: 1-shot {final['acc_1shot']:.1f} vs {EARLIER['1-shot']} "
      f"({d1:+.1f}), 5-shot {final['acc_5shot']:.1f} vs {EARLIER['5-shot']} ({d5:+.1f})")

gpu = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
tv = __import__("torchvision").__version__
runtime = time.time() - extra_start
extra["meta"] = dict(gpu=gpu, torch=torch.__version__, torchvision=tv, seed=SEED,
                     runtime_seconds=runtime, earlier_run=EARLIER, settings=SETTINGS,
                     full_episodes=FULL_EPISODES, check_episodes=CHECK_EPISODES, n_query=N_QUERY)

with open("extra_results.json", "w") as f:
    json.dump(extra, f, indent=2)
rows = extra["A_check"] + extra["A"] + extra["B"] + extra["C"]
fields = list(dict.fromkeys(k for r in rows for k in r))
with open("extra_results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print(f"\nsaved {os.path.abspath('extra_results.json')} and {os.path.abspath('extra_results.csv')}")
print(f"{gpu}, torch {torch.__version__}, torchvision {tv}, extra measurements took {runtime:.0f} s")

A + C. Accuracy on 40 novel species, 600 episodes per setting, 15 queries per class (mean ± 95% interval)
method                                                 5-way 1-shot   5-way 2-shot   5-way 5-shot  5-way 10-shot  5-way 20-shot  10-way 1-shot  10-way 5-shot  20-way 1-shot  20-way 5-shot
Untrained ResNet-18 + prototypes                         36.2 ± 0.8     40.8 ± 0.8     45.6 ± 0.7     48.3 ± 0.7     49.5 ± 0.7     24.5 ± 0.4     31.9 ± 0.4     16.6 ± 0.3     22.6 ± 0.2
ImageNet ResNet-18 + prototypes                          76.0 ± 0.9     89.2 ± 0.6     95.5 ± 0.3     96.8 ± 0.3     97.6 ± 0.2     68.0 ± 0.7     92.0 ± 0.3     60.1 ± 0.4     87.8 ± 0.2
ImageNet ResNet-18 + logistic regression                 83.7 ± 0.8     91.7 ± 0.5     96.1 ± 0.3     97.3 ± 0.2     98.1 ± 0.2     76.9 ± 0.6     92.2 ± 0.3     68.5 ± 0.4     86.9 ± 0.3
ImageNet ResNet-18 + episodic training + prototypes      81.0 ± 0.8     93.9 ± 0.4     97.9 ± 0.2     98.7 ± 0.1     99.2 ± 0.1     74.2 ± 0.6